# Texture Analysis - Client Presentation

Use this notebook to run texture analysis (UMAP StandardScaler, Feature Importance, Rule-Based Thresholds, and Presentation Grid) for **LIMUC** and **TMC** datasets.

**IMPORTANT**: Please modify the paths/directories in the block below according to the dataset locations on your Jupyter server.


In [ ]:
import os
import cv2
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import umap
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import StandardScaler
from PIL import Image, ImageDraw, ImageFont
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# CHANGE DIRECTORIES ACCORDING TO YOUR SERVER
# ==========================================
BASE_DIR = "."  # Change to the root folder path of the Colonomind project
DATA_DIR = os.path.join(BASE_DIR, "data")
REPORTS_DIR = os.path.join(BASE_DIR, "reports")
FIG_DIR = os.path.join(REPORTS_DIR, "figures")

os.makedirs(FIG_DIR, exist_ok=True)

# Raw Image Dataset Directories on Jupyter Server
LIMUC_RAW_DIR = "/home/ubuntu/Colonoscopy/Dataset/LIMUC"
TMC_RAW_DIR = "/home/ubuntu/Colonoscopy/Dataset/TMC-UCM"

FEAT_NAMES = [
    "LL_Mean", "LL_Std", "LL_Var", "LL_Ent", 
    "LH_Mean", "LH_Std", "LH_Var", "LH_Ent",
    "HL_Mean", "HL_Std", "HL_Var", "HL_Ent", 
    "HH_Mean", "HH_Std", "HH_Var", "HH_Ent", "HH_Energy",
    "GLCM_Contrast", "GLCM_Dissimilarity", "GLCM_Homogeneity"
]

print("✅ Modules and Directories are ready.")



In [ ]:
def analyze_and_plot_umap(dataset_name, dl_path, texture_path, labels_path):
    print(f"\n--- Analyzing {dataset_name} ---")
    try:
        dl_features = np.load(dl_path)
        texture_features = np.load(texture_path)
        labels = np.load(labels_path)
    except FileNotFoundError as e:
        print(f"❌ File not found: {e}. Ensure the .npy files are in the correct directory.")
        return None, None, None
        
    subsample_idx = np.arange(dl_features.shape[0])
    if len(subsample_idx) > 10000:
        np.random.seed(42)
        subsample_idx = np.random.choice(subsample_idx, 10000, replace=False)
        
    dl_sub = dl_features[subsample_idx]
    texture_sub = texture_features[subsample_idx]
    labels_sub = labels[subsample_idx]
    
    # StandardScaler
    scaler_dl = StandardScaler()
    dl_sub_scaled = scaler_dl.fit_transform(dl_sub)
    scaler_tex = StandardScaler()
    texture_sub_scaled = scaler_tex.fit_transform(texture_sub)
    
    # UMAP
    print("Computing UMAP embeddings (This may take 1-2 minutes)...")
    reducer_dl = umap.UMAP(n_neighbors=30, min_dist=0.3, random_state=42)
    umap_dl = reducer_dl.fit_transform(dl_sub_scaled)
    
    reducer_texture = umap.UMAP(n_neighbors=30, min_dist=0.3, random_state=42)
    umap_texture = reducer_texture.fit_transform(texture_sub_scaled)
    
    # Random Forest for Feature Importance
    print("Training Classification Models...")
    X_train_tex, X_test_tex, y_train, y_test = train_test_split(texture_features, labels, test_size=0.2, random_state=42, stratify=labels)
    rf_tex = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_tex.fit(X_train_tex, y_train)
    
    return (umap_dl, umap_texture, labels_sub), rf_tex, (dl_features, texture_features, labels)

print("Analysis function is ready to run.")



In [ ]:
def run_analysis_for_dataset(dataset_name, dl_path, tex_path, lbl_path):
    umap_res, rf_model, raw_data = analyze_and_plot_umap(dataset_name, dl_path, tex_path, lbl_path)

    if umap_res:
        umap_dl, umap_texture, labels_sub = umap_res[:3]
        unique_labels = np.unique(labels_sub)
        colors = sns.color_palette("husl", len(unique_labels))
        
        fig, axes = plt.subplots(1, 2, figsize=(16, 7))
        for idx, label in enumerate(unique_labels):
            mask = (labels_sub == label)
            axes[0].scatter(umap_dl[mask, 0], umap_dl[mask, 1], color=colors[idx], label=f'MES {label}', alpha=0.6, s=10)
            axes[1].scatter(umap_texture[mask, 0], umap_texture[mask, 1], color=colors[idx], label=f'MES {label}', alpha=0.6, s=10)
            
        axes[0].set_title(f'{dataset_name} UMAP: Raw Deep Learning Features (Before)', fontsize=14)
        axes[1].set_title(f'{dataset_name} UMAP: Texture Features (After StandardScaler)', fontsize=14)
        axes[0].legend()
        axes[1].legend()
        plt.show()
        
        # Plot Feature Importance
        importances = rf_model.feature_importances_
        indices = np.argsort(importances)[::-1]
        plt.figure(figsize=(12, 6))
        plt.title(f"Feature Importances for {dataset_name} (Texture Analysis)")
        plt.bar(range(len(importances)), importances[indices], align="center")
        plt.xticks(range(len(importances)), [FEAT_NAMES[i] for i in indices], rotation=45, ha='right')
        plt.xlim([-1, len(importances)])
        plt.show()
        
        return umap_res, rf_model, raw_data
    return None, None, None

# Execute LIMUC
limuc_res = run_analysis_for_dataset(
    "LIMUC",
    os.path.join(DATA_DIR, "limuc_features", "limuc_dl_features.npy"),
    os.path.join(DATA_DIR, "limuc_features", "limuc_texture_features.npy"),
    os.path.join(DATA_DIR, "limuc_features", "limuc_labels.npy")
)

# Execute TMC
tmc_res = run_analysis_for_dataset(
    "TMC",
    os.path.join(DATA_DIR, "tmc_features", "tmc_dl_features.npy"),
    os.path.join(DATA_DIR, "tmc_features", "tmc_texture_features.npy"),
    os.path.join(DATA_DIR, "tmc_features", "tmc_labels.npy")
)



In [ ]:
def print_rules(dataset_name, raw_data, rf_model):
    _, texture_features, labels = raw_data
    importances = rf_model.feature_importances_
    top_indices = np.argsort(importances)[::-1][:3] # Top 3 features
    top_features = [FEAT_NAMES[i] for i in top_indices]
    
    df = pd.DataFrame(texture_features, columns=FEAT_NAMES)
    df['Label'] = labels
    
    print(f"=== {dataset_name} Rule-Based Thresholds ===")
    print(f"Top features: {', '.join(top_features)}\n")
    
    for label in np.unique(labels):
        print(f"--- MES {label} ---")
        class_df = df[df['Label'] == label]
        for feat_name in top_features:
            q1 = class_df[feat_name].quantile(0.25)
            q3 = class_df[feat_name].quantile(0.75)
            mean_val = class_df[feat_name].mean()
            print(f"{feat_name}: {q1:.4f} to {q3:.4f} (Mean: {mean_val:.4f})")
        print("")

if limuc_res[0]: print_rules("LIMUC", limuc_res[2], limuc_res[1])
if tmc_res[0]: print_rules("TMC", tmc_res[2], tmc_res[1])



In [ ]:
def load_or_create_image(dataset_name, mes_class):
    clean_label = str(int(float(mes_class)))
    img_path = None
    
    if dataset_name == "LIMUC":
        limuc_paths = {
            "0": "/home/ubuntu/Colonoscopy/Dataset/LIMUC/patient_based_classified_images/1/Mayo 0/UC_patient_1_16.bmp",
            "1": "/home/ubuntu/Colonoscopy/Dataset/LIMUC/patient_based_classified_images/1/Mayo 1/UC_patient_1_11.bmp",
            "2": "/home/ubuntu/Colonoscopy/Dataset/LIMUC/patient_based_classified_images/1/Mayo 2/UC_patient_1_10.bmp",
            "3": "/home/ubuntu/Colonoscopy/Dataset/LIMUC/patient_based_classified_images/10/Mayo 3/UC_patient_10_27.bmp"
        }
        img_path = limuc_paths.get(clean_label)
    else:
        # Hardcode TMC from user's train.txt
        tmc_paths = {
            "0": "/home/ubuntu/Colonoscopy/Dataset/TMC-UCM/images/P02204.JPG",
            "1": "/home/ubuntu/Colonoscopy/Dataset/TMC-UCM/images/P04880.JPG",
            "2": "/home/ubuntu/Colonoscopy/Dataset/TMC-UCM/images/P07855.JPG",
            "3": "/home/ubuntu/Colonoscopy/Dataset/TMC-UCM/images/P10984.JPG"
        }
        img_path = tmc_paths.get(clean_label)
            
    img_arr = None
    if img_path and os.path.exists(img_path):
        try: img_arr = np.array(Image.open(img_path).convert('RGB').resize((256, 256)))
        except: pass
    return img_arr

def plot_presentation_grid(dataset_name, umap_res):
    if not umap_res[0]: return
    
    umap_dl, umap_texture, labels_sub = umap_res[0][:3]
    unique_labels = np.unique(labels_sub)
    colors = sns.color_palette("husl", len(unique_labels))
    
    print(f"\n--- Grid Visualization {dataset_name} ---")
    for label in unique_labels:
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        clean_label = str(int(float(label)))
        
        # 1. Raw Image
        img_arr = load_or_create_image(dataset_name, label)
        if img_arr is not None:
            axes[0].imshow(img_arr)
            axes[0].set_title(f"{dataset_name} Raw Image (MES {clean_label})")
        else:
            img = Image.new('RGB', (256, 256), color=(200, 200, 200))
            d = ImageDraw.Draw(img)
            d.text((50, 100), f"Insert {dataset_name}\nMES {clean_label} Image Here", fill=(50, 50, 50))
            axes[0].imshow(np.array(img))
            axes[0].set_title(f"Image Not Found")
        axes[0].axis('off')
        
        # 2. UMAP Before
        axes[1].scatter(umap_dl[:, 0], umap_dl[:, 1], color='lightgray', alpha=0.3, s=10)
        mask = (labels_sub == label)
        axes[1].scatter(umap_dl[mask, 0], umap_dl[mask, 1], color=colors[int(float(label))], label=f'MES {clean_label}', alpha=0.5, s=15)
        axes[1].set_title('Raw Deep Learning UMAP (Before)')
        axes[1].axis('off')
        
        # 3. UMAP After
        axes[2].scatter(umap_texture[:, 0], umap_texture[:, 1], color='lightgray', alpha=0.3, s=10)
        axes[2].scatter(umap_texture[mask, 0], umap_texture[mask, 1], color=colors[int(float(label))], label=f'MES {clean_label} Cluster', alpha=0.5, s=15)
        
        representive_idx = np.where(mask)[0]
        if len(representive_idx) > 0:
            star_x = umap_texture[representive_idx[0], 0]
            star_y = umap_texture[representive_idx[0], 1]
            # Normal size but with thick black outline for visibility
            axes[2].scatter(star_x, star_y, color=colors[int(float(label))], marker='o', s=60, edgecolor='black', linewidth=1.5, zorder=5, label=f'Current Image')
                
        axes[2].set_title('Texture Analysis UMAP (After)')
        axes[2].legend()
        axes[2].axis('off')
        
        plt.tight_layout()
        plt.show()

# Run the visualization
plot_presentation_grid("LIMUC", limuc_res)
plot_presentation_grid("TMC", tmc_res)



In [ ]:
def export_and_zip_figures(limuc_res, tmc_res, zip_filename="Texture_Analysis_Presentation_Figures"):
    print("Mempersiapkan folder export...")
    export_dir = os.path.join(BASE_DIR, "export_figures")
    os.makedirs(export_dir, exist_ok=True)
    
    datasets = {"LIMUC": limuc_res, "TMC": tmc_res}
    
    for dataset_name, res in datasets.items():
        if not res or not res[0]: continue
        
        umap_dl, umap_texture, labels_sub = res[0][:3]
        rf_model = res[1]
        unique_labels = np.unique(labels_sub)
        colors = sns.color_palette("husl", len(unique_labels))
        
        # 1. Simpan Feature Importance (Bar Chart)
        print(f"Menyimpan Feature Importance untuk {dataset_name}...")
        importances = rf_model.feature_importances_
        indices = np.argsort(importances)[::-1]
        
        fig_feat = plt.figure(figsize=(12, 6))
        plt.title(f"Feature Importances for {dataset_name} (Texture Analysis)")
        plt.bar(range(len(importances)), importances[indices], align="center")
        plt.xticks(range(len(importances)), [FEAT_NAMES[i] for i in indices], rotation=45, ha='right')
        plt.xlim([-1, len(importances)])
        plt.tight_layout()
        plt.savefig(os.path.join(export_dir, f"{dataset_name}_Feature_Importance.png"), dpi=150, bbox_inches='tight')
        plt.close(fig_feat)
        
        # 2. Simpan UMAP Grids
        print(f"Menyimpan UMAP Grids untuk {dataset_name}...")
        for label in unique_labels:
            fig, axes = plt.subplots(1, 3, figsize=(18, 5))
            clean_label = str(int(float(label)))
            
            # Raw Image
            img_arr = load_or_create_image(dataset_name, label)
            if img_arr is not None:
                axes[0].imshow(img_arr)
                axes[0].set_title(f"{dataset_name} Raw Image (MES {clean_label})")
            axes[0].axis('off')
            
            # UMAP Before
            axes[1].scatter(umap_dl[:, 0], umap_dl[:, 1], color='lightgray', alpha=0.3, s=10)
            mask = (labels_sub == label)
            axes[1].scatter(umap_dl[mask, 0], umap_dl[mask, 1], color=colors[int(float(label))], label=f'MES {clean_label}', alpha=0.5, s=15)
            axes[1].set_title('Raw Deep Learning UMAP (Before)')
            axes[1].axis('off')
            
            # UMAP After
            axes[2].scatter(umap_texture[:, 0], umap_texture[:, 1], color='lightgray', alpha=0.3, s=10)
            axes[2].scatter(umap_texture[mask, 0], umap_texture[mask, 1], color=colors[int(float(label))], label=f'MES {clean_label} Cluster', alpha=0.5, s=15)
            
            representive_idx = np.where(mask)[0]
            if len(representive_idx) > 0:
                star_x = umap_texture[representive_idx[0], 0]
                star_y = umap_texture[representive_idx[0], 1]
                axes[2].scatter(star_x, star_y, color=colors[int(float(label))], marker='o', s=60, edgecolor='black', linewidth=1.5, zorder=5, label=f'Current Image')
                    
            axes[2].set_title('Texture Analysis UMAP (After)')
            axes[2].legend()
            axes[2].axis('off')
            
            plt.tight_layout()
            # Save silently instead of showing
            plt.savefig(os.path.join(export_dir, f"{dataset_name}_Grid_MES_{clean_label}.png"), dpi=200, bbox_inches='tight')
            plt.close(fig)
            
    # Zip direktori tersebut
    print("Membungkus ke file ZIP...")
    shutil.make_archive(zip_filename, 'zip', export_dir)
    print(f"✅ Selesai! Semua figure telah disimpan dan di-ZIP.")
    print(f"Silakan download file '{zip_filename}.zip' yang ada di direktori server Jupyter Anda!")

# Jalankan fungsi ekspor
export_and_zip_figures(limuc_res, tmc_res)



---
## Feature Importance Per MES (Log Scale)
Visualisasi nilai fitur tekstur secara rata-rata untuk setiap tingkat keparahan MES (0-3). Fitur yang paling membedakan dilingkari dengan warna merah.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import os
from PIL import Image, ImageDraw
from pathlib import Path

def create_placeholder_endoscopy(text_label, color):
    img = Image.new('RGB', (256, 256), color=(220, 150, 150))
    d = ImageDraw.Draw(img)
    for i in range(10):
        d.arc([(20+i*10, 30+i*5), (200-i*5, 220+i*5)], start=0, end=180, fill=(200, 100, 100), width=3)
    return np.array(img)

def find_sample_image(limuc_root, mes_label):
    limuc_path = Path(limuc_root)
    if not limuc_path.exists():
        return None
    class_mapping = {"0": 0, "1": 1, "2": 2, "3": 3, "Mayo 0": 0, "Mayo 1": 1, "Mayo 2": 2, "Mayo 3": 3}
    for img_path in limuc_path.rglob("*.*"):
        if img_path.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp"]:
            if img_path.parent.name in class_mapping and class_mapping[img_path.parent.name] == mes_label:
                try:
                    img = Image.open(str(img_path)).convert("RGB")
                    return np.array(img.resize((256, 256)))
                except:
                    continue
    return None

def generate_mes_row(ax_img, ax_bar, mes_label, img_data, features, values, badge_color, circle_indices=[]):
    ax_img.imshow(img_data)
    ax_img.axis('off')
    
    badge_rect = patches.Rectangle((0, -40), 256, 40, facecolor=badge_color, edgecolor='none')
    ax_img.add_patch(badge_rect)
    ax_img.text(128, -20, f"MES {mes_label}", color='white', fontweight='bold', fontsize=16, ha='center', va='center')

    x_pos = np.arange(len(features))
    safe_values = np.where(values <= 0, 1e-6, values)
    ax_bar.bar(x_pos, safe_values, color='#1f77b4', edgecolor='white')
    ax_bar.set_yscale('log')
    ax_bar.set_xticks(x_pos)
    ax_bar.set_xticklabels(features, rotation=45, ha='right', fontsize=8)
    ax_bar.set_ylabel("Raw feature value (log scale)", fontsize=8)
    ax_bar.yaxis.grid(True, linestyle='-', which='major', color='lightgrey', alpha=0.5)
    
    for idx in circle_indices:
        if idx < len(x_pos):
            x = x_pos[idx]
            y = safe_values[idx]
            ellipse = patches.Ellipse((x, y), width=1.5, height=y*0.8, edgecolor='red', facecolor='none', lw=2)
            ax_bar.add_patch(ellipse)

# Ganti dengan dataset target (LIMUC atau TMC)
dataset_type = "limuc" # 'limuc' or 'tmc'
limuc_root = "/raid/D13K48009/texture/LIMUC"
data_dir = f"data/{dataset_type}_features"
tex_path = os.path.join(data_dir, f"{dataset_type}_texture_features.npy")
lbl_path = os.path.join(data_dir, f"{dataset_type}_labels.npy")

features_data, labels_data = None, None
if os.path.exists(tex_path) and os.path.exists(lbl_path):
    features_data = np.load(tex_path)
    labels_data = np.load(lbl_path)

features = ["LL_Mean", "LL_Std", "LL_Var", "LL_Entropy", "LH_Mean", "LH_Std", "LH_Var", "LH_Entropy",
            "HL_Mean", "HL_Std", "HL_Var", "HL_Entropy", "HH_Mean", "HH_Std", "HH_Var", "HH_Entropy",
            "HH_Energy", "GLCM_Contrast", "GLCM_Dissimilarity", "GLCM_Homogeneity"]

fig = plt.figure(figsize=(14, 12))
from matplotlib.gridspec import GridSpec
gs = GridSpec(4, 5, figure=fig, width_ratios=[1, 1.5, 1.5, 1.5, 1.5])
colors = ['#2ca02c', '#bcbd22', '#ff7f0e', '#d62728']

for mes in range(4):
    ax_img = fig.add_subplot(gs[mes, 0])
    ax_bar = fig.add_subplot(gs[mes, 1:])
    img_data = find_sample_image(limuc_root, mes)
    if img_data is None: img_data = create_placeholder_endoscopy(f"MES {mes}", colors[mes])
        
    if features_data is not None:
        mes_mask = (labels_data == mes)
        if np.sum(mes_mask) > 0:
            mean_vals = np.mean(np.abs(features_data[mes_mask]), axis=0)
            vals = mean_vals[:len(features)] if len(mean_vals) >= len(features) else np.pad(mean_vals, (0, len(features)-len(mean_vals)))
        else:
            vals = np.random.uniform(10, 100000, size=len(features))
    else:
        vals = np.random.uniform(10, 100000, size=len(features))
        
    generate_mes_row(ax_img, ax_bar, str(mes), img_data, features, vals, colors[mes], circle_indices=[2, 18])

plt.tight_layout()
plt.show()

---
## UMAP Visualization (Before vs After)
Membandingkan sebaran distribusi *latent space* UMAP sebelum ekstraksi tekstur (menggunakan raw Deep Learning features) dan sesudah menggunakan fitur tekstur (3-Channel DWT + GLCM).

In [ ]:
import seaborn as sns
import umap
from sklearn.preprocessing import StandardScaler

dl_path = os.path.join(data_dir, f"{dataset_type}_dl_features.npy")

if os.path.exists(dl_path) and features_data is not None:
    dl_features = np.load(dl_path)
    
    # Subsample agar UMAP berjalan lebih cepat di Notebook
    max_samples = min(5000, len(labels_data))
    np.random.seed(42)
    sub_idx = np.random.choice(len(labels_data), max_samples, replace=False)
    
    dl_sub = dl_features[sub_idx]
    tex_sub = features_data[sub_idx]
    lbl_sub = labels_data[sub_idx]
    
    print("Menghitung UMAP... Mohon tunggu sebentar.")
    umap_dl = umap.UMAP(n_neighbors=30, min_dist=0.3, random_state=42).fit_transform(StandardScaler().fit_transform(dl_sub))
    umap_tex = umap.UMAP(n_neighbors=30, min_dist=0.3, random_state=42).fit_transform(StandardScaler().fit_transform(tex_sub))
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    colors_map = sns.color_palette("husl", 4)
    
    for i in range(4):
        mask = (lbl_sub == i)
        axes[0].scatter(umap_dl[mask, 0], umap_dl[mask, 1], color=colors_map[i], label=f'MES {i}', alpha=0.6, s=15)
        axes[1].scatter(umap_tex[mask, 0], umap_tex[mask, 1], color=colors_map[i], label=f'MES {i}', alpha=0.6, s=15)
        
    axes[0].set_title("Before: Raw Deep Learning Features", fontsize=14)
    axes[1].set_title("After: Texture Analysis Features", fontsize=14)
    
    for ax in axes:
        ax.axis('off')
        ax.legend()
        
    plt.tight_layout()
    plt.show()
else:
    print("Fitur Deep Learning atau Texture Features tidak ditemukan di path:", data_dir)